# Seahorse Citibike Case Study

This notebook installs Seahorse, downloads the `yahya021/citibike-stpp`
Hugging Face dataset, trains a fast parametric baseline (`PoissonGMM`) and a
neural model (`DeepSTPP`), evaluates both, and visualizes the results.

It is designed for a fresh CPU-only Colab runtime. The dataset has thousands of
sequences, so the notebook uses a capped subset for a quick first run.

## Install and import

This Colab installs Seahorse directly from the GitHub branch that contains
the notebook, then imports the public Python API.

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/YahyaAalaila/seahorse.git"
REPO_REF = "codex/citibike-colab-case-study"

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "seahorse", "seahorse-stpp"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", f"git+{REPO_URL}@{REPO_REF}"],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == "seahorse" or module_name.startswith("seahorse."):
        del sys.modules[module_name]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from seahorse import DeepSTPP, PoissonGMM
from seahorse.data import load_dataset


## Load the Hugging Face dataset

`load_dataset` accepts a Hugging Face repo id when the repo exposes Seahorse
JSONL split files. Citibike is the lightest public case-study dataset in this
set, so it is a good Colab default.

In [ ]:
DATASET_ID = "yahya021/citibike-stpp"
FULL_SPLITS = load_dataset(DATASET_ID)

FULL_SPLIT_SUMMARY = pd.DataFrame(
    [
        {
            "split": name,
            "sequences": len(records),
            "events": sum(len(seq["times"]) for seq in records),
        }
        for name, records in FULL_SPLITS.items()
    ]
).sort_values("split")

display(FULL_SPLIT_SUMMARY)

## Use a CPU-friendly subset

The full Citibike splits are available in `FULL_SPLITS`. For a short Colab case
study, keep the first non-trivial sequences from each split. Increase or remove
these caps for a real run.

In [ ]:
MAX_TRAIN_SEQUENCES = 64
MAX_VAL_SEQUENCES = 16
MAX_TEST_SEQUENCES = 16


def keep_sequences(records, limit):
    kept = [seq for seq in records if len(seq.get("times", [])) >= 3]
    return kept if limit is None else kept[:limit]


train = keep_sequences(FULL_SPLITS["train"], MAX_TRAIN_SEQUENCES)
val = keep_sequences(FULL_SPLITS["val"], MAX_VAL_SEQUENCES)
test = keep_sequences(FULL_SPLITS["test"], MAX_TEST_SEQUENCES)

summary = pd.DataFrame(
    [
        {"split": "train", "sequences": len(train), "events": sum(len(seq["times"]) for seq in train)},
        {"split": "val", "sequences": len(val), "events": sum(len(seq["times"]) for seq in val)},
        {"split": "test", "sequences": len(test), "events": sum(len(seq["times"]) for seq in test)},
    ]
)
display(summary)

print("Example sequence keys:", sorted(train[0].keys()))
print("First five event times:", train[0]["times"][:5])
print("First five locations:", train[0]["locations"][:5])

## Inspect event trajectories

Each record is a sequence of bike-trip departure events. Locations are already
in the dataset coordinate system expected by Seahorse.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for idx, sequence in enumerate(train[:20]):
    locs = np.asarray(sequence["locations"], dtype=float)
    ax.plot(locs[:, 0], locs[:, 1], marker="o", linewidth=0.9, markersize=2.5, alpha=0.45)

ax.set_title("Citibike training trajectories")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
plt.show()

## Create two models

`PoissonGMM` is the fast baseline. `DeepSTPP` is the neural STPP model. The
DeepSTPP overrides keep the network small so the notebook can run on CPU.

In [ ]:
MAX_EPOCHS = 1
BATCH_SIZE = 16

baseline_overrides = {
    "data": {"num_workers": 0},
}

deep_overrides = {
    "data": {"num_workers": 0, "normalize": True},
    "model": {
        "hidden_dim": 16,
        "encoder": {
            "num_heads": 1,
            "num_layers": 1,
            "dropout": 0.0,
        },
        "decoder": {
            "seq_len": 8,
            "lookahead": 1,
            "num_points": 8,
            "n_layers": 1,
        },
    },
}

models = {
    "PoissonGMM": PoissonGMM(device="cpu", seed=42, config_overrides=baseline_overrides),
    "DeepSTPP": DeepSTPP(device="cpu", seed=42, config_overrides=deep_overrides),
}

{name: estimator.preset for name, estimator in models.items()}

## Fit

Both estimators train on the same Citibike subset.

In [ ]:
%%capture training_log
for name, estimator in models.items():
    estimator.fit(
        train,
        val,
        test,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        dataset_id="citibike-stpp-colab-subset",
    )

In [ ]:
print("Training completed for:", ", ".join(models))
print(training_log.stdout[-1600:])

## Evaluate

The Python API reports likelihood-oriented metrics on the held-out test subset.

In [ ]:
rows = []
for name, estimator in models.items():
    scores = estimator.evaluate(test)
    rows.append({"model": name, **scores})

results = pd.DataFrame(rows).set_index("model").sort_values("test_nll")
display(results)

## Visualize model scores

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
results["test_nll"].plot(kind="bar", ax=ax, color=["tab:blue", "tab:orange"])
ax.set_title("Citibike test negative log likelihood")
ax.set_ylabel("lower is better")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.show()

## Sample next events with DeepSTPP

`predict_next` samples candidate next events for held-out prefixes from the test
sequences. This gives a compact qualitative view of the fitted neural model.

In [ ]:
prediction_contexts = test[:4]
samples = models["DeepSTPP"].predict_next(prediction_contexts, n_samples=8)

print("next_times shape:", samples["next_times"].shape)
print("next_locations shape:", samples["next_locations"].shape)
print("sampling backend:", samples["sampling_backend"])

## Visualize predictive samples

In [ ]:
context_idx = 0
sample_times = samples["next_times"][context_idx]
sample_locs = samples["next_locations"][context_idx]
true_time = samples["true_next_times"][context_idx]
true_loc = samples["true_next_locations"][context_idx]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].hist(sample_times, bins=8, alpha=0.75, color="tab:blue")
axes[0].axvline(true_time, color="black", linestyle="--", label="true next time")
axes[0].set_title("DeepSTPP sampled next times")
axes[0].set_xlabel("time")
axes[0].legend(loc="best")

axes[1].scatter(sample_locs[:, 0], sample_locs[:, 1], alpha=0.75, label="sampled next locations")
axes[1].scatter([true_loc[0]], [true_loc[1]], marker="x", s=90, color="black", label="true next location")
axes[1].set_title("DeepSTPP sampled next locations")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_aspect("equal", adjustable="box")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

## Next steps

- Increase `MAX_EPOCHS` for a meaningful experiment.
- Increase or remove the sequence caps to train on the full Citibike splits.
- Swap `PoissonGMM` for `HawkesGMM` when you want a history-dependent baseline.
- Use the CLI benchmark notebook when comparing many presets, seeds, or datasets.